In [1]:
import pandas as pd
import requests

import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
import glob
import os

In [2]:
import re

def get_key_from_environment(file_path: str, key: str) -> str | None:
    """
    Extracts a key from Angular's environment.ts file.

    Args:
        file_path: Path to environment.ts
        key: The key name (e.g., "apiKey")

    Returns:
        The value as a string, or None if not found.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")



In [3]:
header = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

from datetime import datetime
from dateutil.relativedelta import relativedelta

scale = 12
start_date = datetime(2020, 9, 1)   # Sept 2024
end_date = datetime(2025, 8, 1)     # Aug 2025

# Loop over months
date = start_date
while date <= end_date:
    date_str = date.strftime("%Y-%m")
    url = f"https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale{scale:03d}&date={date_str}"
    
    print(f"Fetching {url} ...")
    res = requests.get(url, headers=header)
    
    if res.status_code == 200:
        file = f"./data/spi{scale:03d}_{date_str}.tif"
        with open(file, "wb") as f:
            f.write(res.content)
    
    # Move to next month
    date += relativedelta(months=1)

Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2020-09 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2020-10 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2020-11 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2020-12 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2021-01 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2021-02 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2021-03 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2021-04 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale012&date=2021-05 ...
Fetching h

In [4]:
#statewide
shapefile = "../public/Coastline.shp"
raster_folder = "./data"

# Load shapefile and dissolve to one feature (statewide)
gdf = gpd.read_file(shapefile)
gdf_statewide = gdf.dissolve()  # merges all polygons into one

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    
    # Compute zonal stats for the dissolved geometry
    stats = zonal_stats(
        vectors=gdf_statewide,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=-9999
    )
    
    records.append({
        "date": date,
        "value": stats[0]["mean"]
    })


# Pivot to wide format: one row (statewide), months as columns
df = pd.DataFrame(records).set_index("date").T
df.insert(0, "state", "statewide")  # add proper state column
df.to_csv(f"../public/statewide_spi{scale}.csv", index=False)

print(df)

date       state   2020-09   2020-10   2020-11  2020-12  2021-01   2021-02  \
value  statewide  0.561331  0.479813  0.606564  0.46056  0.26788  0.221295   

date    2021-03  2021-04   2021-05  ...   2024-11   2024-12   2025-01  \
value  0.372087  0.35898  0.294948  ...  0.024834 -0.161592 -0.150209   

date    2025-02   2025-03   2025-04   2025-05   2025-06  2025-07   2025-08  
value -0.269035 -0.095722 -0.171698 -0.439413 -0.401593 -0.40669 -0.875027  

[1 rows x 61 columns]


In [5]:
shapefile = "../public/hawaii_climate_divisions.shp"

raster_folder = "./data"

gdf = gpd.read_file(shapefile)

id_col = "name"  

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    
    # Zonal statistics
    stats = zonal_stats(
        vectors=gdf,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=-9999
    )
    
    # Store results
    for div, s in zip(gdf[id_col], stats):
        records.append({
            "division": div,
            "date": date,
            "mean_spi": s["mean"]
        })

df = pd.DataFrame(records)
df = df.pivot(index="division", columns="date", values="mean_spi")
df = df.reindex(sorted(df.columns), axis=1)
df.to_csv(f"../public/division_spi{scale}.csv")



In [ ]:
shapefile = "../public/Coastline.shp"

raster_folder = "./data"

gdf = gpd.read_file(shapefile)

id_col = "isle"  

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    
    # Zonal statistics
    stats = zonal_stats(
        vectors=gdf,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=-9999
    )
    
    # Store results
    for div, s in zip(gdf[id_col], stats):
        records.append({
            "division": div,
            "date": date,
            "mean_spi": s["mean"]
        })

df = pd.DataFrame(records)
df = df.pivot(index="division", columns="date", values="mean_spi")
df = df.reindex(sorted(df.columns), axis=1)
df.to_csv(f"../public/island_spi{scale}.csv")
print(df)


date        2020-09   2020-10   2020-11   2020-12   2021-01   2021-02  \
division                                                                
Hawaiʻi    0.688695  0.551178  0.735163  0.709207  0.417340  0.386377   
Kahoolawe -0.386276 -0.236260 -0.306980 -0.667407 -0.395885 -0.501158   
Kauaʻi     1.494776  1.414218  1.368370  0.870750  0.693205  0.798598   
Lānaʻi    -0.400608 -0.266984 -0.290955 -0.617361 -0.369720 -0.409273   
Maui       0.069301  0.092361  0.153222 -0.104608 -0.212703 -0.371695   
Molokaʻi  -0.251183 -0.053256 -0.154516 -0.407647 -0.221559 -0.412569   
Oʻahu      0.088644  0.063065  0.198830 -0.178094 -0.129581 -0.220154   

date        2021-03   2021-04   2021-05   2021-06  ...   2024-11   2024-12  \
division                                           ...                       
Hawaiʻi    0.383028  0.424143  0.361874  0.352446  ...  0.156255  0.029552   
Kahoolawe -0.130724 -0.328709 -0.414707 -0.487549  ... -0.204780 -0.281347   
Kauaʻi     0.753086  0.757100 

: 

In [ ]:
shapefile = "../public/watersheds.shp"

raster_folder = "./data"

gdf = gpd.read_file(shapefile)

id_col = "isle"  

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    
    # Zonal statistics
    stats = zonal_stats(
        vectors=gdf,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=-9999
    )
    
    # Store results
    for div, s in zip(gdf[id_col], stats):
        records.append({
            "division": div,
            "date": date,
            "mean_spi": s["mean"]
        })

df = pd.DataFrame(records)
df = df.pivot(index="division", columns="date", values="mean_spi")
df = df.reindex(sorted(df.columns), axis=1)
df.to_csv(f"../public/watershed_spi{scale}.csv")
print(df)


In [ ]:
folder = "./data"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):  # ensures it's a file
        os.remove(file_path)